In [10]:
input='/Users/thomas/Desktop/github_repos/industry_time_series/src/research/timestamps_collection.csv'
output='res.csv', 
root_folder = '/Users/thomas/Data/Data_sensors'

In [ ]:
"""
Vehicle Event Analysis: Extract peak timestamps from boundary sensor pairs

For each vehicle event:
1. Adjust timestamp (add offset for weighing system delay)
2. Find and load appropriate hourly CSV file
3. Extract time window around vehicle passage
4. Detect first peak and dominant peak for each boundary sensor
5. Output sensor pair timestamps to CSV

Output CSV structure:
- Rows: vehicle events (v1, v2, ...)
- Columns: for each sensor pair, 4 timestamps:
    - sensor1_first_peak_time
    - sensor1_peak_time
    - sensor2_first_peak_time
    - sensor2_peak_time
"""

import pandas as pd
import numpy as np
import polars as pl
from datetime import datetime, timedelta
import os
import argparse
from pathlib import Path

# Import from provided code
from src.shared.bridge_model import load_bridge
from src.shared.config import position_csv, threshold_csv, delimiter


# =========================================================
# FILE DISCOVERY
# =========================================================
def find_csv_file(root_folder, date_str, hour):
    """Find the hourly CSV file for given date and hour."""
    hour1 = hour + 1
    formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
    csv_folder = os.path.join(root_folder, date_str, "csv_acc")

    for fname in os.listdir(csv_folder):
        if (
            fname.startswith(f"M001_{formatted_date}_{hour:02d}-00-00_")
            and f"_int-{hour1}_" in fname
            and fname.endswith("_th.csv")
        ):
            return os.path.join(csv_folder, fname)

    raise FileNotFoundError(f"No matching CSV file found for {date_str} hour {hour}")


# =========================================================
# DATA EXTRACTION
# =========================================================
def get_only_interested_duration(df, sensor_columns, time_column, start_time, duration_mins):
    """Extract specific time window from dataframe."""
    print(f"  Selecting columns and converting to pandas...")
    
    # Select only necessary columns
    sampled_df = df.select([time_column] + sensor_columns).to_pandas()
    print(f"  Sampled data shape: {sampled_df.shape}")

    # Convert time column
    sampled_df[time_column] = pd.to_datetime(
        sampled_df[time_column], 
        format='%Y/%m/%d %H:%M:%S:%f', 
        errors="coerce", 
        exact=False
    )
    
    # Define time window
    start_time = pd.to_datetime(start_time)
    end_time = start_time + pd.Timedelta(minutes=duration_mins)

    # Filter to specific time frame
    sampled_df = sampled_df[
        (sampled_df[time_column] >= start_time) & 
        (sampled_df[time_column] <= end_time)
    ]
    
    print(f"  Filtered to {len(sampled_df)} samples in {duration_mins}-min window")
    return sampled_df


def _get_filtered_mask(sensor_series: pd.Series, threshold: float, sample_period: int) -> np.ndarray:
    """
    Return a boolean mask of the same length as sensor_series.
    True  → sample is inside an active window (|value| >= threshold,
            plus the sample_period tail after each crossing).
    False → sample is outside every active window.
    """
    n    = len(sensor_series)
    mask = np.zeros(n, dtype=bool)
    vals = sensor_series.to_numpy()

    i = 0
    while i < n:
        if np.abs(vals[i]) >= threshold:
            start = i
            end   = min(i + sample_period, n)

            # Extend the window as long as the signal keeps crossing threshold
            while i < end:
                if np.abs(vals[i]) >= threshold:
                    end = min(i + sample_period, n)
                i += 1

            mask[start:end] = True
        else:
            i += 1

    return mask


def find_sensor_peaks(
    df: pd.DataFrame,
    sensor_ids: list,
    time_column: str,
    threshold: float,
    sample_period: int
):
    """
    Find one first peak and one dominant peak per event window.
 
    Returns:
        Dict mapping sensor_id -> list of
        (first_peak_time, dominant_peak_time, first_peak_amplitude, dominant_peak_amplitude)
    """
    results = {}
 
    for sensor_id in sensor_ids:
        sensor_series = pd.Series(df[sensor_id].values, dtype=float)
        raw_signal    = sensor_series.to_numpy()
        time_series   = df[time_column]
        events        = []
 
        # Step 1: get event mask
        mask = _get_filtered_mask(sensor_series, threshold, sample_period)
 
        # Step 2: find contiguous True regions
        diff   = np.diff(mask.astype(int))
        starts = np.where(diff == 1)[0] + 1
        ends   = np.where(diff == -1)[0] + 1
 
        if mask[0]:
            starts = np.insert(starts, 0, 0)
        if mask[-1]:
            ends = np.append(ends, len(mask))
 
        # Step 3: find extrema within each window
        for win_start, win_end in zip(starts, ends):
            seg = raw_signal[win_start:win_end]
            seg_len = len(seg)
 
            extrema = []
 
            # Boundary peak check
            if (seg[0] > 0 and seg[0] > seg[1]) or \
               (seg[0] < 0 and seg[0] < seg[1]):
                extrema.append(0)
 
            # Interior extrema
            for k in range(1, seg_len - 1):
                if seg[k] >= seg[k - 1] and seg[k] >= seg[k + 1]:
                    extrema.append(k)
                elif seg[k] <= seg[k - 1] and seg[k] <= seg[k + 1]:
                    extrema.append(k)
 
            # Fallback if no extrema
            if not extrema:
                fallback = int(np.argmax(np.abs(seg)))
                extrema.append(fallback)
 
            # First peak
            first_peak_local = extrema[0]
 
            # Dominant peak
            dominant_local = max(extrema, key=lambda idx: abs(seg[idx]))
 
            # Convert to global indices
            fp_idx = win_start + first_peak_local
            dp_idx = win_start + dominant_local
 
            events.append((
                time_series.iloc[fp_idx],          # first_peak_time
                time_series.iloc[dp_idx],          # dominant_peak_time
                float(raw_signal[fp_idx]),         # first_peak_amplitude
                float(raw_signal[dp_idx]),         # dominant_peak_amplitude
            ))
 
        results[sensor_id] = events
        print(f"    {sensor_id}: {len(events)} event(s)")
 
    return results


# =========================================================
# VEHICLE EVENT PROCESSING
# =========================================================
def process_vehicle_event(
    vehicle_id: str,
    timestamp_str: str,
    root_folder: str,
    boundary_sensors: list,
    threshold: float,
    sample_period: int,
    time_offset_minutes: int,
    duration_minutes: int
):
    """
    Process a single vehicle event.
    
    Args:
        vehicle_id: Vehicle identifier (e.g., "v1", "v2")
        timestamp_str: Original timestamp from weighing system (format: "YYYY-MM-DD HH:MM:SS")
        root_folder: Root data folder
        boundary_sensors: List of boundary sensor IDs
        threshold: Peak detection threshold
        sample_period: Sample period for event windowing
        time_offset_minutes: Minutes to add to timestamp (weighing system delay)
        duration_minutes: Duration of analysis window
        
    Returns:
        Dictionary with sensor pair data
    """
    print(f"\n{'='*60}")
    print(f"Processing {vehicle_id}: {timestamp_str}")
    print(f"{'='*60}")
    
    # Step 1: Adjust timestamp
    original_time = pd.to_datetime(timestamp_str)
    adjusted_time = original_time + timedelta(minutes=time_offset_minutes)
    
    print(f"Original timestamp: {original_time}")
    print(f"Adjusted timestamp: {adjusted_time} (+{time_offset_minutes} min)")
    
    # Step 2: Find CSV file
    date_str = adjusted_time.strftime("%Y%m%d")
    hour = adjusted_time.hour
    
    print(f"Looking for CSV: date={date_str}, hour={hour}")
    csv_path = find_csv_file(root_folder, date_str, hour)
    print(f"Found: {csv_path}")
    
    # Step 3: Load and filter data
    print(f"Loading CSV with Polars...")
    df_full = pl.read_csv(csv_path, separator=';')
    
    print(f"Extracting {duration_minutes}-min window...")
    df_window = get_only_interested_duration(
        df_full,
        sensor_columns=boundary_sensors,
        time_column='time',
        start_time=adjusted_time,
        duration_mins=duration_minutes
    )
    
    # Remove DC offset (mean centering)
    for sensor in boundary_sensors:
        df_window[sensor] = df_window[sensor] - df_window[sensor].mean()
    
    # Step 4: Find peaks for all boundary sensors
    print(f"Detecting peaks (threshold={threshold}, sample_period={sample_period})...")
    peak_results = find_sensor_peaks(
        df_window,
        sensor_ids=boundary_sensors,
        time_column='time',
        threshold=threshold,
        sample_period=sample_period
    )
    
    # Step 5: Extract data for sensor pairs
    # Create pairs from consecutive boundary sensors
    sensor_pairs = [(boundary_sensors[i], boundary_sensors[i+1]) 
                    for i in range(len(boundary_sensors) - 1)]
    
    result = {'vehicle_id': vehicle_id, 'original_timestamp': timestamp_str}
    
    for pair_idx, (sensor1, sensor2) in enumerate(sensor_pairs, start=1):
        pair_name = f"pair_{pair_idx}"
        
        # Get first event from each sensor (assuming one vehicle passage)
        events1 = peak_results.get(sensor1, [])
        events2 = peak_results.get(sensor2, [])
        
        if events1 and events2:
            # Take the first event from each sensor
            first_peak_time1, peak_time1, _, _ = events1[0]
            first_peak_time2, peak_time2, _, _ = events2[0]
            
            result[f"{pair_name}_{sensor1}_first_peak"] = first_peak_time1
            result[f"{pair_name}_{sensor1}_peak"] = peak_time1
            result[f"{pair_name}_{sensor2}_first_peak"] = first_peak_time2
            result[f"{pair_name}_{sensor2}_peak"] = peak_time2
        else:
            # No events detected
            result[f"{pair_name}_{sensor1}_first_peak"] = None
            result[f"{pair_name}_{sensor1}_peak"] = None
            result[f"{pair_name}_{sensor2}_first_peak"] = None
            result[f"{pair_name}_{sensor2}_peak"] = None
            
            print(f"  WARNING: No events for pair {pair_idx} ({sensor1}, {sensor2})")
    
    return result


# =========================================================
# MAIN
# =========================================================
def main():
    parser = argparse.ArgumentParser(
        description="Analyze vehicle events across bridge boundary sensors"
    )
    
    parser.add_argument(
        '--input',
        type=str,
        required=True,
        help='CSV file with vehicle timestamps (columns: vehicle_id, timestamp)'
    )
    
    parser.add_argument(
        '--output',
        type=str,
        required=True,
        help='Output CSV path for results'
    )
    
    parser.add_argument(
        '--root_folder',
        type=str,
        required=True,
        help='Root folder containing date-organized sensor data'
    )
    
    parser.add_argument(
        '--threshold',
        type=float,
        default=0.002,
        help='Peak detection threshold (default: 0.002)'
    )
    
    parser.add_argument(
        '--sample_period',
        type=int,
        default=300,
        help='Sample period for event windowing (default: 300)'
    )
    
    parser.add_argument(
        '--time_offset',
        type=int,
        default=2,
        help='Minutes to add to weighing system timestamp (default: 2)'
    )
    
    parser.add_argument(
        '--duration',
        type=int,
        default=5,
        help='Analysis window duration in minutes (default: 5)'
    )
    
    args = parser.parse_args()
    
    # Load bridge model
    print("Loading bridge model...")
    bridge = load_bridge(position_csv, threshold_csv, delimiter=delimiter)
    junctions = bridge.find_boundaries()
    
    # Get all boundary sensor IDs (in physical order)
    boundary_sensors = [s for j in junctions for s in j.sensor_ids()]
    print(f"Boundary sensors ({len(boundary_sensors)}): {boundary_sensors}")
    
    # Load vehicle timestamps
    print(f"\nLoading vehicle timestamps from: {args.input}")
    vehicles_df = pd.read_csv(args.input)
    
    if 'vehicle_id' not in vehicles_df.columns or 'timestamp' not in vehicles_df.columns:
        raise ValueError("Input CSV must have 'vehicle_id' and 'timestamp' columns")
    
    print(f"Found {len(vehicles_df)} vehicle events")
    
    # Process each vehicle
    all_results = []
    
    for idx, row in vehicles_df.iterrows():
        vehicle_id = row['vehicle_id']
        timestamp = row['timestamp']
        
        try:
            result = process_vehicle_event(
                vehicle_id=vehicle_id,
                timestamp_str=timestamp,
                root_folder=args.root_folder,
                boundary_sensors=boundary_sensors,
                threshold=args.threshold,
                sample_period=args.sample_period,
                time_offset_minutes=args.time_offset,
                duration_minutes=args.duration
            )
            all_results.append(result)
            print(f"✓ {vehicle_id} processed successfully")
            
        except Exception as e:
            print(f"✗ ERROR processing {vehicle_id}: {e}")
            # Still add a row with NaN values
            result = {'vehicle_id': vehicle_id, 'original_timestamp': timestamp}
            all_results.append(result)
    
    # Save results
    print(f"\n{'='*60}")
    print(f"Saving results to: {args.output}")
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(args.output, index=False)
    print(f"Saved {len(results_df)} vehicle events")
    print(f"Columns: {list(results_df.columns)}")
    print(f"{'='*60}")
    

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --input INPUT --output OUTPUT --root_folder
                             ROOT_FOLDER [--threshold THRESHOLD]
                             [--sample_period SAMPLE_PERIOD]
                             [--time_offset TIME_OFFSET] [--duration DURATION]
ipykernel_launcher.py: error: the following arguments are required: --input, --output, --root_folder


SystemExit: 2

/Users/thomas/Desktop/github_repos/industry_time_series/venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3554: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from datetime import datetime, timedelta
import os
import argparse
# =========================================================
# FILE DISCOVERY
# =========================================================
def find_csv_file(root_folder, date_str, hour):
    hour1 = hour + 1
    formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
    csv_folder = os.path.join(root_folder, date_str, "csv_acc")

    for fname in os.listdir(csv_folder):
        if (
            fname.startswith(f"M001_{formatted_date}_{hour:02d}-00-00_")
            and f"_int-{hour1}_" in fname
            and fname.endswith("_th.csv")
        ):
            return os.path.join(csv_folder, fname)

    raise FileNotFoundError("No matching CSV file found.")
from src.shared.bridge_model import *
from src.shared.config import *

Bridge = load_bridge(position_csv, threshold_csv, delimiter=delimiter)
junctions = Bridge.find_boundaries()

all_boundary_ids = [s for j in junctions for s in j.sensor_ids()]
all_boundary_nums = [s for j in junctions for s in j.sensor_numbers()]

print(all_boundary_ids)
#print(all_boundary_nums)

sensor_columns = all_boundary_ids
# =========================================================
# DATA EXTRACTION
# =========================================================
def extract_5min_data(csv_path, sensor_id, start_time_str):
    df = pd.read_csv(csv_path, sep=';')
    df['time'] = pd.to_datetime(
        df['time'],
        format='%Y/%m/%d %H:%M:%S:%f',
        errors="coerce",
        exact=False
    )

    h, m, s = map(int, start_time_str.split(':'))
    base_date = df['time'].iloc[0].date()

    start_dt = datetime.combine(base_date, datetime.min.time().replace(hour=h, minute=m, second=s))
    end_dt = start_dt + timedelta(minutes=5)

    seg = df[(df['time'] >= start_dt) & (df['time'] < end_dt)]

    if seg.empty:
        raise ValueError("No data in interval")

    return seg[sensor_id].values, start_dt, end_dt
def get_only_interested_duration(df, sensor_columns, time_column, start_time, duration_mins):
    memory_usage()

    # Select only necessary columns and sample at specified interval
    sampled_df = (df.select([time_column] + sensor_columns)  # Keep relevant columns
    #.collect() used if you are scanning the file
    .to_pandas()
    )
    print(f"Sampled data shape: {sampled_df.shape}")
    memory_usage()

    sampled_df[time_column] = pd.to_datetime(sampled_df[time_column], format='%Y/%m/%d %H:%M:%S:%f', errors="coerce", exact=False)
    

    #PLOT ONLY duration we want to analyse 
    start_time = pd.to_datetime(start_time)
    

    end_time = start_time + pd.Timedelta(minutes=duration_mins)

    #limit to the specific time frame
    sampled_df = sampled_df[(sampled_df[time_column]>=start_time)&(sampled_df[time_column]<=end_time)]
    
    memory_usage()

    return sampled_df
def _get_filtered_mask(sensor_series: pd.Series, threshold: float, sample_period: int) -> np.ndarray:
    """
    Return a boolean mask of the same length as sensor_series.
    True  → sample is inside an active window (|value| >= threshold,
            plus the sample_period tail after each crossing).
    False → sample is outside every active window.
    """
    n    = len(sensor_series)
    mask = np.zeros(n, dtype=bool)
    vals = sensor_series.to_numpy()   # work on a plain numpy array for speed

    i = 0
    while i < n:
        if np.abs(vals[i]) >= threshold:
            start = i
            end   = min(i + sample_period, n)

            # Extend the window as long as the signal keeps crossing threshold
            while i < end:
                if np.abs(vals[i]) >= threshold:
                    end = min(i + sample_period, n)
                i += 1

            mask[start:end] = True
        else:
            i += 1

    return mask






def find_sensor_peaks(
    df: pd.DataFrame,
    sensor_ids: str,
    time_column: str,
    threshold: float,
    sample_period
):
    """
    Find one first peak and one dominant peak per event window.
 
    Uses _get_filtered_mask() to group nearby threshold crossings into
    single event windows (via sample_period extension), then within each
    contiguous window finds extrema on the RAW signal (not abs):
      - first_peak:    first extremum (local max or min) in the window,
                       including the very first sample (boundary check)
      - dominant_peak: extremum with the largest absolute amplitude
 
    Peak detection uses raw signed values so that a positive peak followed
    by a larger negative swing doesn't get missed.
 
    Args:
        df: DataFrame with time and sensor columns (DC already removed)
        sensor_ids: List of sensor column names
        time_column: Name of time column
        threshold: Absolute amplitude threshold for event window detection
        sample_period: Samples to extend window after each crossing
                       (same parameter as filterby_threshold, default 300)
 
    Returns:
        Dict mapping sensor_id -> list of
        (first_peak_time, dominant_peak_time, first_peak_amplitude, dominant_peak_amplitude)
    """
    results = {}
 
    for sensor_id in sensor_ids:
        sensor_series = pd.Series(df[sensor_id].values, dtype=float)
        raw_signal    = sensor_series.to_numpy()
        time_series   = df[time_column]               # pandas Series
        events        = []
 
        # Step 1: get event mask using existing _get_filtered_mask
        mask = _get_filtered_mask(sensor_series, threshold, sample_period)
 
        # Step 2: find contiguous True regions (each = one event window)
        diff   = np.diff(mask.astype(int))
        starts = np.where(diff == 1)[0] + 1       # False->True transitions
        ends   = np.where(diff == -1)[0] + 1       # True->False transitions
 
        # Handle edge cases: mask starts or ends with True
        if mask[0]:
            starts = np.insert(starts, 0, 0)
        if mask[-1]:
            ends = np.append(ends, len(mask))
 
        # Step 3: within each event window, find extrema on RAW signal
        for win_start, win_end in zip(starts, ends):
            seg = raw_signal[win_start:win_end]
            seg_len = len(seg)
 
            # if seg_len < 2:
            #     # Single sample — use it directly
            #     p_idx = win_start
            #     events.append((
            #         time_series.iloc[p_idx],
            #         time_series.iloc[p_idx],
            #         float(raw_signal[p_idx]),
            #         float(raw_signal[p_idx]),
            #     ))
            #     continue
 
            # --- Collect all extrema (local max + local min) on raw signal ---
            extrema = []
 
            # k=0: boundary peak — signal already turning at window start.
            # Positive crossing that immediately drops: seg[0] > 0 and seg[0] > seg[1]
            # Negative crossing that immediately rises: seg[0] < 0 and seg[0] < seg[1]
            # If signal is still growing in magnitude, k=0 is NOT a peak.
            if (seg[0] > 0 and seg[0] > seg[1]) or \
               (seg[0] < 0 and seg[0] < seg[1]):
                extrema.append(0)
 
            # k=1..seg_len-2: interior extrema
            for k in range(1, seg_len - 1):
                if seg[k] >= seg[k - 1] and seg[k] >= seg[k + 1]:
                    extrema.append(k)   # local max
                elif seg[k] <= seg[k - 1] and seg[k] <= seg[k + 1]:
                    extrema.append(k)   # local min
 
            # Fallback: no extrema found (monotonic signal), use sample
            # with largest absolute amplitude
            if not extrema:
                fallback = int(np.argmax(np.abs(seg)))
                extrema.append(fallback)
 
            # --- First peak: first extremum in the window ---
            first_peak_local = extrema[0]
 
            # --- Dominant peak: extremum with largest |amplitude| ---
            dominant_local = max(extrema, key=lambda idx: abs(seg[idx]))
 
            # Convert to global indices, return signed amplitudes
            fp_idx = win_start + first_peak_local
            dp_idx = win_start + dominant_local
 
            events.append((
                time_series.iloc[fp_idx],          # first_peak_time
                time_series.iloc[dp_idx],          # dominant_peak_time
                float(raw_signal[fp_idx]),         # first_peak_amplitude (signed)
                float(raw_signal[dp_idx]),         # dominant_peak_amplitude (signed)
            ))
 
        results[sensor_id] = events
        print(f"[PEAKS] {sensor_id}: {len(events)} event(s) "
              f"(threshold={threshold}, sample_period={sample_period})")
 
    return results


In [4]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('res.csv')

df.describe()


,vehicle_id,original_timestamp,03091010_x_first_peak,03091010_x_peak,03091203_x_first_peak,03091203_x_peak,03091153_x_first_peak,03091153_x_peak,0309101F_x_first_peak,0309101F_x_peak,...,03091032_x_first_peak,03091032_x_peak,0309102D_x_first_peak,0309102D_x_peak,03091007_x_first_peak,03091007_x_peak,03091043_x_first_peak,03091043_x_peak,030911EC_x_first_peak,030911EC_x_peak
count,3,3,3,3,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3
unique,3,3,3,3,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3
top,v1,2025-07-03 00:18:00,2025-03-07 00:20:15.350,2025-03-07 00:20:15.600,2025-03-07 00:20:15.360,2025-03-07 00:20:15.600,2025-03-07 00:20:15.360,2025-03-07 00:20:19.360,2025-03-07 00:20:15.370,2025-03-07 00:20:19.360,...,2025-03-07 00:20:37.760,2025-03-07 00:20:37.790,2025-03-07 00:19:52.740,2025-03-07 00:19:52.850,2025-03-07 00:20:41.680,2025-03-07 00:20:43.440,2025-03-07 00:20:41.720,2025-03-07 00:20:43.270,2025-03-07 00:19:52.880,2025-03-07 00:19:52.880
freq,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [ ]:
"""
Visualize Vehicle Event Analysis Results

Reads the output CSV from vehicle_event_analysis.py and creates:
1. Timeline plot - Shows when each vehicle was detected by each sensor
2. Sensor progression plot - Shows vehicle movement through sensors over time

Usage:
    python visualize_vehicle_events.py --input results.csv --output_dir plots/
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import argparse
from pathlib import Path
import re


def extract_sensor_columns(df):
    """
    Extract sensor column information from the dataframe.
    
    Returns:
        sensor_list: Ordered list of unique sensor IDs
        first_peak_cols: Dict mapping sensor_id -> column name for first_peak
        peak_cols: Dict mapping sensor_id -> column name for dominant peak
    """
    # Find all columns ending with '_first_peak' or '_peak'
    first_peak_cols = {}
    peak_cols = {}
    sensor_order = []
    
    for col in df.columns:
        if col.endswith('_first_peak'):
            # Extract sensor ID (everything before '_first_peak')
            sensor_id = col.replace('_first_peak', '')
            first_peak_cols[sensor_id] = col
            if sensor_id not in sensor_order:
                sensor_order.append(sensor_id)
        elif col.endswith('_peak') and not col.endswith('_first_peak'):
            # Extract sensor ID (everything before '_peak')
            sensor_id = col.replace('_peak', '')
            peak_cols[sensor_id] = col
    
    print(f"Found {len(sensor_order)} unique sensors")
    return sensor_order, first_peak_cols, peak_cols


def plot_timeline(df, sensor_list, first_peak_cols, peak_cols, output_path):
    """
    Create timeline plot showing when each vehicle was detected by each sensor.
    
    X-axis: Absolute time (datetime)
    Y-axis: Vehicle events (v1, v2, v3...)
    Markers: Blue dots for first_peak, red dots for dominant_peak
    """
    print("\n" + "="*60)
    print("Generating Timeline Plot")
    print("="*60)
    
    fig, ax = plt.subplots(figsize=(16, max(8, len(df) * 0.5)))
    
    # Y-axis positions for each vehicle
    y_positions = {vehicle_id: idx for idx, vehicle_id in enumerate(df['vehicle_id'])}
    
    # Collect all timestamps for axis limits
    all_times = []
    
    # Plot each vehicle's detections
    for idx, row in df.iterrows():
        vehicle_id = row['vehicle_id']
        y_pos = y_positions[vehicle_id]
        
        # Plot first_peak detections (blue)
        for sensor_id in sensor_list:
            if sensor_id in first_peak_cols:
                col_name = first_peak_cols[sensor_id]
                timestamp = row[col_name]
                
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    all_times.append(time_obj)
                    ax.plot(time_obj, y_pos, 'o', color='blue', markersize=6, alpha=0.6)
        
        # Plot dominant_peak detections (red)
        for sensor_id in sensor_list:
            if sensor_id in peak_cols:
                col_name = peak_cols[sensor_id]
                timestamp = row[col_name]
                
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    all_times.append(time_obj)
                    ax.plot(time_obj, y_pos, 's', color='red', markersize=5, alpha=0.6)
    
    # Configure Y-axis
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['vehicle_id'])
    ax.set_ylabel('Vehicle Event', fontsize=12, fontweight='bold')
    
    # Configure X-axis
    if all_times:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    ax.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax.set_title('Vehicle Detection Timeline Across Sensors', fontsize=14, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
               markersize=8, label='First Peak', alpha=0.6),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='red', 
               markersize=7, label='Dominant Peak', alpha=0.6)
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ Timeline plot saved to: {output_path}")
    plt.close()


def plot_sensor_progression(df, sensor_list, first_peak_cols, peak_cols, output_path):
    """
    Create sensor progression plot showing vehicle movement through sensors.
    
    X-axis: Sensor index (0, 1, 2, 3...)
    Y-axis: Time relative to first detection (seconds)
    Lines: Solid for first_peak, dashed for dominant_peak
    Different color per vehicle
    """
    print("\n" + "="*60)
    print("Generating Sensor Progression Plot")
    print("="*60)
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    # Color map for vehicles
    colors = plt.cm.tab20(np.linspace(0, 1, len(df)))
    
    # Plot each vehicle
    for idx, row in df.iterrows():
        vehicle_id = row['vehicle_id']
        color = colors[idx]
        
        # Collect first_peak times
        first_peak_times = []
        first_peak_indices = []
        
        for sensor_idx, sensor_id in enumerate(sensor_list):
            if sensor_id in first_peak_cols:
                col_name = first_peak_cols[sensor_id]
                timestamp = row[col_name]
                
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    first_peak_times.append(time_obj)
                    first_peak_indices.append(sensor_idx)
        
        # Collect dominant_peak times
        dominant_peak_times = []
        dominant_peak_indices = []
        
        for sensor_idx, sensor_id in enumerate(sensor_list):
            if sensor_id in peak_cols:
                col_name = peak_cols[sensor_id]
                timestamp = row[col_name]
                
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    dominant_peak_times.append(time_obj)
                    dominant_peak_indices.append(sensor_idx)
        
        # Convert to relative times (seconds from first detection)
        if first_peak_times:
            reference_time = min(first_peak_times)
            
            # First peak line (solid)
            relative_first = [(t - reference_time).total_seconds() for t in first_peak_times]
            ax.plot(first_peak_indices, relative_first, '-o', 
                   color=color, linewidth=2, markersize=6,
                   label=f'{vehicle_id} (first peak)', alpha=0.7)
            
            # Dominant peak line (dashed)
            if dominant_peak_times:
                relative_dominant = [(t - reference_time).total_seconds() for t in dominant_peak_times]
                ax.plot(dominant_peak_indices, relative_dominant, '--s', 
                       color=color, linewidth=1.5, markersize=5,
                       label=f'{vehicle_id} (dominant peak)', alpha=0.5)
        else:
            print(f"  WARNING: No detections for {vehicle_id}")
    
    # Configure X-axis
    ax.set_xticks(range(len(sensor_list)))
    ax.set_xticklabels(sensor_list, rotation=90, ha='right', fontsize=8)
    ax.set_xlabel('Sensor (Physical Order)', fontsize=12, fontweight='bold')
    
    # Configure Y-axis
    ax.set_ylabel('Time Since First Detection (seconds)', fontsize=12, fontweight='bold')
    ax.set_title('Vehicle Progression Through Sensors', fontsize=14, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Add legend (outside plot if many vehicles)
    if len(df) <= 10:
        ax.legend(loc='best', fontsize=9)
    else:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ Sensor progression plot saved to: {output_path}")
    plt.close()


# =========================================================
# VISUALIZATION FUNCTIONS
# =========================================================
def extract_sensor_columns(df):
    """Extract sensor column information from results dataframe."""
    first_peak_cols = {}
    peak_cols = {}
    sensor_order = []
    
    for col in df.columns:
        if col.endswith('_first_peak'):
            sensor_id = col.replace('_first_peak', '')
            first_peak_cols[sensor_id] = col
            if sensor_id not in sensor_order:
                sensor_order.append(sensor_id)
        elif col.endswith('_peak') and not col.endswith('_first_peak'):
            sensor_id = col.replace('_peak', '')
            peak_cols[sensor_id] = col
    
    return sensor_order, first_peak_cols, peak_cols
 
 
def plot_timeline(df, sensor_list, first_peak_cols, peak_cols, output_path):
    """
    Timeline plot showing when each vehicle was detected by each sensor.
    
    Y-axis: Vehicles, X-axis: Time
    Blue dots = first_peak, Red squares = dominant_peak
    """
    print("\n  Generating timeline plot...")
    
    fig, ax = plt.subplots(figsize=(16, max(8, len(df) * 0.5)))
    
    y_positions = {vehicle_id: idx for idx, vehicle_id in enumerate(df['vehicle_id'])}
    all_times = []
    
    # Plot detections
    for idx, row in df.iterrows():
        vehicle_id = row['vehicle_id']
        y_pos = y_positions[vehicle_id]
        
        # First peaks (blue)
        for sensor_id in sensor_list:
            if sensor_id in first_peak_cols:
                timestamp = row[first_peak_cols[sensor_id]]
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    all_times.append(time_obj)
                    ax.plot(time_obj, y_pos, 'o', color='blue', markersize=6, alpha=0.6)
        
        # Dominant peaks (red)
        for sensor_id in sensor_list:
            if sensor_id in peak_cols:
                timestamp = row[peak_cols[sensor_id]]
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    all_times.append(time_obj)
                    ax.plot(time_obj, y_pos, 's', color='red', markersize=5, alpha=0.6)
    
    # Configure axes
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['vehicle_id'])
    ax.set_ylabel('Vehicle Event', fontsize=12, fontweight='bold')
    
    if all_times:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    ax.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax.set_title('Vehicle Detection Timeline', fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
               markersize=8, label='First Peak', alpha=0.6),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='red', 
               markersize=7, label='Dominant Peak', alpha=0.6)
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Timeline saved: {output_path}")
    plt.close()
 
 
def plot_sensor_progression(df, sensor_list, first_peak_cols, peak_cols, output_path):
    """
    Sensor progression plot showing vehicle movement through sensors.
    
    X-axis: Sensor index, Y-axis: Time since first detection (seconds)
    Solid lines = first_peak, Dashed lines = dominant_peak
    """
    print("\n  Generating sensor progression plot...")
    
    fig, ax = plt.subplots(figsize=(16, 10))
    colors = plt.cm.tab20(np.linspace(0, 1, len(df)))
    
    # Plot each vehicle
    for idx, row in df.iterrows():
        vehicle_id = row['vehicle_id']
        color = colors[idx]
        
        # Collect first_peak times
        first_peak_times = []
        first_peak_indices = []
        
        for sensor_idx, sensor_id in enumerate(sensor_list):
            if sensor_id in first_peak_cols:
                timestamp = row[first_peak_cols[sensor_id]]
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    first_peak_times.append(time_obj)
                    first_peak_indices.append(sensor_idx)
        
        # Collect dominant_peak times
        dominant_peak_times = []
        dominant_peak_indices = []
        
        for sensor_idx, sensor_id in enumerate(sensor_list):
            if sensor_id in peak_cols:
                timestamp = row[peak_cols[sensor_id]]
                if pd.notna(timestamp):
                    time_obj = pd.to_datetime(timestamp)
                    dominant_peak_times.append(time_obj)
                    dominant_peak_indices.append(sensor_idx)
        
        # Convert to relative times
        if first_peak_times:
            reference_time = min(first_peak_times)
            
            # First peak line (solid)
            relative_first = [(t - reference_time).total_seconds() for t in first_peak_times]
            ax.plot(first_peak_indices, relative_first, '-o', 
                   color=color, linewidth=2, markersize=6,
                   label=f'{vehicle_id} (first)', alpha=0.7)
            
            # Dominant peak line (dashed)
            if dominant_peak_times:
                relative_dominant = [(t - reference_time).total_seconds() for t in dominant_peak_times]
                ax.plot(dominant_peak_indices, relative_dominant, '--s', 
                       color=color, linewidth=1.5, markersize=5,
                       label=f'{vehicle_id} (dominant)', alpha=0.5)
    
    # Configure axes
    ax.set_xticks(range(len(sensor_list)))
    ax.set_xticklabels(sensor_list, rotation=90, ha='right', fontsize=8)
    ax.set_xlabel('Sensor (Physical Order)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Time Since First Detection (seconds)', fontsize=12, fontweight='bold')
    ax.set_title('Vehicle Progression Through Sensors', fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Legend
    if len(df) <= 10:
        ax.legend(loc='best', fontsize=9)
    else:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Progression saved: {output_path}")
    plt.close()
 
 
def generate_visualizations(results_csv, output_dir):
    """Generate timeline and progression plots from results CSV."""
    print("\n" + "="*60)
    print("GENERATING VISUALIZATIONS")
    print("="*60)
    
    # Load results
    df = pd.read_csv(results_csv)
    print(f"Loaded {len(df)} vehicle events")
    
    # Extract sensor info
    sensor_list, first_peak_cols, peak_cols = extract_sensor_columns(df)
    print(f"Found {len(sensor_list)} sensors")
    
    # Create output directory
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Generate plots
    timeline_path = output_dir / 'vehicle_timeline.png'
    plot_timeline(df, sensor_list, first_peak_cols, peak_cols, timeline_path)
    
    progression_path = output_dir / 'vehicle_progression.png'
    plot_sensor_progression(df, sensor_list, first_peak_cols, peak_cols, progression_path)
    
    print("\n" + "="*60)
    print("Visualizations complete!")
    print("="*60)
 



def main():
    parser = argparse.ArgumentParser(
        description="Visualize vehicle event analysis results"
    )
    
    parser.add_argument(
        '--input',
        type=str,
        required=True,
        help='Input CSV file from vehicle_event_analysis.py'
    )
    
    parser.add_argument(
        '--output_dir',
        type=str,
        default='.',
        help='Output directory for plots (default: current directory)'
    )
    
    parser.add_argument(
        '--timeline',
        action='store_true',
        help='Generate timeline plot'
    )
    
    parser.add_argument(
        '--progression',
        action='store_true',
        help='Generate sensor progression plot'
    )
    
    args = parser.parse_args()
    
    # If no plot flags specified, generate both
    if not args.timeline and not args.progression:
        args.timeline = True
        args.progression = True
    
    # Create output directory
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("="*60)
    print("Vehicle Event Visualization")
    print("="*60)
    print(f"Input file: {args.input}")
    print(f"Output directory: {output_dir}")
    
    # Load data
    print("\nLoading data...")
    df = pd.read_csv(args.input)
    print(f"Loaded {len(df)} vehicle events")
    print(f"Total columns: {len(df.columns)}")
    
    # Extract sensor information
    sensor_list, first_peak_cols, peak_cols = extract_sensor_columns(df)
    
    # Generate plots
    if args.timeline:
        timeline_path = output_dir / 'vehicle_timeline.png'
        plot_timeline(df, sensor_list, first_peak_cols, peak_cols, timeline_path)
    
    if args.progression:
        progression_path = output_dir / 'vehicle_progression.png'
        plot_sensor_progression(df, sensor_list, first_peak_cols, peak_cols, progression_path)
    
    print("\n" + "="*60)
    print("Visualization complete!")
    print("="*60)


if __name__ == "__main__":
    main()